# Task II: Code Review (LangGraph Port)

This notebook generates paired traces for code review tasks.
Agents read source code, analyze it, and produce review reports.

## Setup
- Uses MCP filesystem server for tool calls
- Generates paired traces with shared execution_id
- LEP: FC1.3 (Step Repetition) - reviewer loops on same file
- LEP: FC2.2 (Fail to Ask for Clarification) - reviewer picks wrong version

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.') / 'src'))

from agentgraph import (
    JSONLTraceParser,
    EntityGraphBuilder,
    GraphEncoder,
    ExportManager,
)
from benchmarks import CodeReviewTask, MockLLMBackend, TraceConfig
from pipeline import TraceAnalyzer, analyze_pairs, validate_graph_correctness

print("Library loaded for code review task.")


ModuleNotFoundError: No module named 'agentgraph'

In [ ]:
# --- Configuration ---
WORKSPACE = Path("workspace")
OUTPUT_DIR = Path("output")
TRACE_DIR = Path("traces")
for d in [WORKSPACE, OUTPUT_DIR, TRACE_DIR]:
    d.mkdir(exist_ok=True)

# --- Generate paired traces ---
task = CodeReviewTask(WORKSPACE)
llm = MockLLMBackend()
config = TraceConfig(
    task_name="code_review",
    max_events_per_run=120,
    min_events_per_run=90,
)

traces = task.generate_traces(llm, config)
benign = traces["benign"]
malignant = traces["malignant"]

print(f"Benign:   {benign.trace_id} ({benign.num_events} events)")
print(f"Malignant: {malignant.trace_id} ({malignant.num_events} events)")


In [ ]:
# --- Build graphs ---
builder = EntityGraphBuilder()
benign_g = builder.build(benign)
mal_g = builder.build(malignant)

print(f"Benign graph:   {benign_g.num_nodes} nodes, {benign_g.num_edges} edges")
print(f"Malignant graph: {mal_g.num_nodes} nodes, {mal_g.num_edges} edges")

# --- Analyze ---
analyzer = TraceAnalyzer()
diff = analyzer.compare_traces(benign, malignant)
print(f"Content change: {diff.content_change_pct:.2%}")
print(f"LEP codes: {diff.lep_codes}")


In [ ]:
# --- Export ---
encoder = GraphEncoder()
static_data, temporal_data = encoder.encode(
    [benign_g, mal_g], labels=[0.0, 1.0]
)
exporter = ExportManager(OUTPUT_DIR)
exporter.export_dyglib_dataset([benign_g, mal_g], "code_review")
exporter.save_torch(static_data, "code_review_graphs.pt")

import json
for variant, trace in traces.items():
    path = TRACE_DIR / f"trace_{trace.trace_id}.jsonl"
    with open(path, "w") as f:
        for event in trace.events:
            f.write(json.dumps(event.to_dict()) + "\n")

print("Export complete.")
